# Brazilian BBQ v2 — avaliação do Bode 7B por logprobs

Refatoração do `BBQ.ipynb`. Principais mudanças:

1. **Segurança**: sem token HuggingFace hardcoded — assume `huggingface-cli login` prévio ou `HF_TOKEN` no ambiente.
2. **Avaliação por logprobs**: em vez de geração livre + parser regex, um único forward pass compara os logits do primeiro token entre as letras A/B/C (com variantes de tokenização detectadas automaticamente, ex. `A` vs `▁A`). Elimina respostas INVALID e o parser inteiro. A geração livre é mantida apenas como auditoria (`raw_generation`) nos primeiros N exemplos.
3. **Permutação tripla**: cada exemplo lógico vira 3 variantes com rotação cíclica das opções (unknown em C, A e B), permitindo medir viés posicional. `correct_option`/`biased_option` são recalculados por permutação.
4. **Métricas novas** (ver `metrics.py`): bias scores do BBQ (Parrish et al. 2022), viés posicional, `position_consistency`, flip rate, unknown rate por contexto, e IC 95% por bootstrap sobre `logical_id`.
5. **Resultados organizados** em `results/run_<timestamp>_<model_tag>_<fase>/` com parquet + csv, checkpoint/retomada, `token_usage.json` e `REPORT.md`.

O conteúdo dos cenários/templates em português é **idêntico** ao original — só a infraestrutura de avaliação mudou. O código vive em `bbq_v2_lib.py`, `metrics.py` e `run_experiment.py`; este notebook orquestra.

In [ ]:
# Dependências (rode uma vez se necessário)
# %pip install -q -U torch --index-url https://download.pytorch.org/whl/cu124
# %pip install -q -U transformers accelerate sentencepiece safetensors pandas pyarrow tqdm matplotlib

import os
# Login: NUNCA hardcode o token. Ou rode `huggingface-cli login` no terminal,
# ou exporte HF_TOKEN antes de abrir o Jupyter.
if "HF_TOKEN" in os.environ:
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
else:
    print("Assumindo login prévio via huggingface-cli login.")

In [ ]:
import importlib
import bbq_v2_lib as lib
import metrics as M
import run_experiment as R
importlib.reload(lib); importlib.reload(M); importlib.reload(R)

import torch
print("CUDA:", torch.cuda.is_available())

## 1. Geração do benchmark (sem modelo)

`build_logical_examples` reproduz exatamente a expansão original (6.000 exemplos lógicos = 2 categorias × 30 cenários × 25 pares × 4 condições); `expand_permutations` triplica com rotação cíclica das opções.

In [ ]:
df_templates = lib.build_base_templates()
df_logical = lib.build_logical_examples(df_templates)
df_expanded = lib.expand_permutations(df_logical)
print(len(df_templates), "templates |", len(df_logical), "exemplos lógicos |", len(df_expanded), "expandidos")

# Sanidade: as 3 permutações de um mesmo logical_id têm contexto idêntico e opções rotacionadas
one = df_expanded[df_expanded.logical_id == df_logical.logical_id.iloc[0]].sort_values("perm_index")
one[["perm_index","option_A","option_B","option_C","unknown_position","correct_option","biased_option"]]

## 2. Execução

`R.run_phase(fase)` faz tudo: cria `results/run_.../`, salva config e dados, roda o scoring com checkpoint, calcula métricas e token usage.

- **smoke**: 30 exemplos lógicos (×3 = 90) — valida o pipeline;
- **pilot**: 600 lógicos balanceados (×3 = 1.800) — diagnóstico do viés posicional e concordância logprob × geração;
- **full**: todos os 6.000 lógicos (×3 = 18.000) — só rode com GPU e tempo estimado a partir do piloto.

In [ ]:
PHASE = "smoke"   # "smoke" | "pilot" | "full"
run_dir, df_results, model, tokenizer = R.run_phase(PHASE)

In [ ]:
# Reaproveite o modelo carregado para a próxima fase, ex.:
# run_dir, df_results, model, tokenizer = R.run_phase("pilot", model=model, tokenizer=tokenizer)

## 3. Métricas e relatório

As tabelas já foram salvas em `run_dir`. Abaixo, inspeção rápida e geração do `REPORT.md` com figuras.

In [ ]:
import pandas as pd
overall = pd.read_csv(run_dir / "metrics_overall.csv")
by_pos = pd.read_csv(run_dir / "metrics_by_position.csv")
display(overall.T)
display(by_pos)

In [ ]:
import make_report
importlib.reload(make_report)
make_report.build_report(run_dir)
print((run_dir / "REPORT.md").read_text()[:3000])